In [1]:
import os
import requests
import json
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [ ]:
spark = SparkSession.builder \
    .appName("PUBG_Telemetry_Silver_Layer") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()
print("Spark Session initialized successfully with PostgreSQL driver!")

:: loading settings :: url = jar:file:/Users/youngeddieb/igdb-analytics-pipeline/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/youngeddieb/.ivy2.5.2/cache
The jars for the packages stored in: /Users/youngeddieb/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fc9d26e5-433f-4fa3-a30c-2d8ab53c33d8;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
:: resolution report :: resolve 66ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframework#checker-qual;3.31.0 from central in [default]
	org.postgresql#postgresql;42.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evic

Spark Session initialized successfully with PostgreSQL driver!


In [3]:
%store -r telemetry_url

In [4]:
response = requests.get(telemetry_url)
json_data = response.json()


In [14]:
print(json_data)

[{'_T': 'LogMatchDefinition', 'MatchId': 'match.bro.tutorialatoz.pc-2018-41.steam.solo.ru.2026.05.09.23.5cc9dd84-d35a-4fd8-8ab4-7c77f7543722', 'PingQuality': '', '_D': '2026-05-09T23:55:39.2085814Z'}, {'_T': 'LogItemEquip', 'character': {'name': 'RevivingTarget', 'teamId': 0, 'health': 100, 'location': {'x': 28200, 'y': 23900, 'z': 162.61463928222656}, 'ranking': 0, 'individualRanking': 0, 'accountId': '', 'isInBlueZone': False, 'isInRedZone': False, 'inSpecialZone': 'None', 'isInVehicle': False, 'zone': [], 'type': 'user', 'isDBNO': False}, 'item': {'itemId': 'Item_Weapon_HK416_C', 'stackCount': 1, 'category': 'Weapon', 'subCategory': 'Main', 'attachedItems': []}, 'common': {'isGame': 0}, '_D': '2026-05-09T23:54:48.626Z'}, {'_T': 'LogItemEquip', 'character': {'name': 'RevivingTarget', 'teamId': 0, 'health': 100, 'location': {'x': 28200, 'y': 23900, 'z': 162.61463928222656}, 'ranking': 0, 'individualRanking': 0, 'accountId': '', 'isInBlueZone': False, 'isInRedZone': False, 'inSpecialZo

In [23]:
json_strings = [json.dumps(record) for record in json_data]
rdd = spark.sparkContext.parallelize(json_strings)
telemetry_url_df  = spark.read.json(rdd)

In [21]:
telemetry_url_df.show(5)

+--------------------+-----------+--------------------+------------------+---------+--------------+--------+----------+--------+---------------------+-------------------+--------------------+----------+---------+------+------+-----------+---------+--------------------+--------------------+---------+----------+------------+-----------+-----------+----------------+--------------------+-------+---------------+----------+-------+--------+------------+-------+------+------+--------+---------+
|             MatchId|PingQuality|                  _D|                _T|accountId|allWeaponStats|attackId|attackType|attacker|blueZoneCustomOptions|cameraViewBehaviour|           character|characters|childItem|common|dBNOId|elapsedTime|fireCount|fireWeaponStackCount|gameResultOnFinished|gameState|healAmount|isCustomGame|isEventMode|isLedgeGrab|isVaultOnVehicle|                item|mapName|numAlivePlayers|parentItem|reviver|teamSize|useTraumaBag|vehicle|victim|weapon|weaponId|weatherId|
+-------------

In [24]:
# Save the DataFrame as a Parquet file
telemetry_url_df.write.mode("overwrite").parquet("data/telemetry_data.parquet")

print("Successfully saved to Parquet!")

26/05/10 23:21:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Successfully saved to Parquet!
